[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/data-engineering-from-scratch-to-advanced/blob/main/notebooks/topics/08_engineering-toolbox.ipynb)

*Data Engineering from Scratch to Advanced · Notebook 8 of 21*

# Engineering Toolbox

The everyday skills around the data: Linux commands, Git, Docker, cron, testing, CI/CD and environments. Each idea is modelled in Python, so you understand what the tools are really doing.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** These tools aren't extras. They're how your code survives contact with other people and production.

**What interviewers really test here:** Reading logs in the shell, Git workflow, Docker layers and how you test pipelines.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. The terminal: your daily tool *(Basics)*
- [ ] 2. Scheduling with cron *(Basics)*
- [ ] 3. Git: how teams work on code *(Core)*
- [ ] 4. Docker: the same everywhere *(Core)*
- [ ] 5. Testing and CI/CD *(Core)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline
import math
import numpy as np
import matplotlib.pyplot as plt
PLAIN = True

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Environments

| Environment | Purpose | Data |
| --- | --- | --- |
| Local | write code | tiny sample |
| Dev | try changes together | synthetic or masked |
| Staging | rehearse a release | production-like |
| Production | the real thing | real, access controlled |

---
# Part 2: Lessons, step by step

## Lesson 1: The terminal: your daily tool

*Level: Basics*

Servers have no mouse. When a job fails at 3 am, you'll log in and read logs. Commands do one small thing each, and a pipe `|` joins them, passing one's output to the next.



| Command | Does |
| --- | --- |
| `ls -lh`, `cd`, `pwd` | look around |
| `head`, `tail -f app.log` | see the start or end; -f follows live |
| `grep -i error app.log` | find lines |
| `wc -l` | count lines |
| `cut -d, -f2`, `awk` | pick columns |
| `sort \| uniq -c \| sort -rn` | count and rank values |
| `du -sh *`, `df -h` | what's using disk space? |
| `ps`, `top`, `kill` | what's running? |
| `chmod`, `chown` | permissions |
| `ssh`, `scp` | connect, and copy to servers |



**A real example:** which errors are most common today?



```python
grep ERROR app.log | cut -d' ' -f4 | sort | uniq -c | sort -rn | head -5
```



You can answer that in 10 seconds without writing a script. In this topic you rebuild these tools in Python, to see what they do.



**Shell scripts:** always start with `set -euo pipefail`. Otherwise a failed step gets ignored and the script carries on, reporting success. Check **exit codes**: 0 means success, and anything else means failure.



**Permissions:** 640 means rw- r-- ---: the owner reads and writes, the group reads, others get nothing. Each digit adds read 4, write 2 and execute 1.

**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q2](#q2), [Q3](#q3), [Q4](#q4), [Q7](#q7), [Q8](#q8), [Q11](#q11), [Q10](#q10). Try one now, or carry on and do them in Part 3.

## Lesson 2: Scheduling with cron

*Level: Basics*

Cron runs commands on a schedule, written as five fields:



```text
┌ minute (0-59)
│ ┌ hour (0-23)
│ │ ┌ day of month
│ │ │ ┌ month
│ │ │ │ ┌ day of week (0 = Sunday)
0 2 * * *    every day at 02:00
*/15 * * * * every 15 minutes
0 2 * * 1    Mondays at 02:00
```



A job with schedule 02:30, checked at 03:00, next runs tomorrow at 02:30.



**Cron is enough for one simple job, but it isn't an orchestrator.** It knows nothing about dependencies, retries, backfills or whether the job succeeded. That's why teams move to Airflow, which comes later in this course.



Remember that cron usually runs in the server's time zone, so check it, or set it to UTC.

**✏️ Your turn:** you now have what you need for [Q12](#q12), [Q13](#q13). Try one now, or carry on and do them in Part 3.

## Lesson 3: Git: how teams work on code

*Level: Core*

Git keeps every version of your code. **Each commit is a snapshot pointing to its parent**, so history is a graph.



```text
a ─ b ─ c          main
     └─ d ─ e      my-feature
```



The commits new on `my-feature` are those reachable from e, but not from c: d and e.



```python
git switch -c fix-dedupe       # new branch
git add transform.py
git commit -m "Keep latest row per customer"
git push -u origin fix-dedupe  # then open a pull request
git pull --rebase origin main  # catch up with main
```



**A pull request** lets teammates review your change, and lets automated tests run, before it reaches main.



**Merge conflicts** are less scary once you see the logic. Git compares three versions: the shared base, yours and theirs. If only one side changed a line, take that change. If both changed it differently, a person has to decide.



**Rules that keep you out of trouble:**



- Small commits with clear messages.
- **Never commit secrets.** Git history keeps them even after you delete the line, so a leaked key must be rotated. Use tools like gitleaks, which catch `API_KEY = "abc"` before it lands.
- Use `.gitignore` for data files and environments.
- Pin dependency versions, so builds can be reproduced.



**Versions:** in 1.4.2, a bug fix gives 1.4.3, a new feature 1.5.0 and a breaking change 2.0.0.

**✏️ Your turn:** you now have what you need for [Q14](#q14), [Q20](#q20), [Q5](#q5), [Q16](#q16), [Q9](#q9). Try one now, or carry on and do them in Part 3.

## Lesson 4: Docker: the same everywhere

*Level: Core*

"It works on my machine" stops being a problem with **containers**. A container bundles your code with its exact dependencies, and it runs the same on a laptop or in production. It's lighter than a virtual machine, as it shares the host's kernel.



```python
FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .          # dependencies first
RUN pip install -r requirements.txt
COPY src/ ./src                  # code last
CMD ["python", "-m", "src.main"]
```



```python
docker build -t my-job .
docker run --rm -e DB_URL=... my-job
```



**Why that order?** Each line is a cached layer, and changing one rebuilds it and everything after it. With steps of 5 s (base), 100 s (install) and 2 s (code):



| Order | Change code, rebuild takes |
| --- | --- |
| install, then code | **2 s** |
| code, then install | 102 s |



**Configuration comes from the environment,** so one image runs in dev and production. Fail immediately when something required is missing, rather than halfway through a run. **Never bake secrets into an image.**



**Kubernetes** runs many containers across machines, restarting failed ones and scaling them. You should understand pods, and resource requests and limits, as many Spark and Airflow platforms run on it.

**✏️ Your turn:** you now have what you need for [Q15](#q15), [Q6](#q6). Try one now, or carry on and do them in Part 3.

## Lesson 5: Testing and CI/CD

*Level: Core*

Data code is often untested, which is why data is often wrong. Your QA background is a real advantage here.



| Test | Checks | Speed |
| --- | --- | --- |
| Unit | one transform on tiny input | milliseconds |
| Integration | steps together, on a test database | seconds to minutes |
| Data quality | real data in production | each run |



```python
import pytest

@pytest.mark.parametrize("rows,expected", [
    ([], 0),                          # empty input!
    ([{"amt": 1}, {"amt": 2}], 3),
])
def test_total(rows, expected):
    assert total(rows) == expected
```



**Make code testable by injecting dependencies.** A load function that accepts a `fetch` function can be tested with a fake one, with no network at all.



**CI (continuous integration)** runs checks on every pull request. **CD** deploys when they pass. **Put cheap checks first.** Lint takes 5 s and fails 10% of the time; tests take 60 s and fail 20%. Running lint first costs an expected 5 + 0.9 × 60 = **59 s** until you stop. Running tests first costs 60 + 0.8 × 5 = 64 s.



```python
# .github/workflows/ci.yml
on: pull_request
jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - run: pip install -r requirements.txt
      - run: ruff check . && pytest
```

**✏️ Your turn:** you now have what you need for [Q17](#q17), [Q18](#q18), [Q19](#q19). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

**▶ Run this cell before the questions.** It sets up the data and tools they share.

In [2]:
import re


<a id="q1"></a>
### Q1. Build grep in Python

🟢 Warm-up · *Search lines*

Return lines containing a word, ignoring case if asked, with their line numbers from 1.

**Examples**

- `lines = ['ok', 'ERROR x', 'error y'], word = 'error', ignore_case = True` → `[[2, 'ERROR x'], [3, 'error y']]`
- `lines = ['ok', 'ERROR x'], word = 'error', ignore_case = False` → `[]`

In [ ]:
def grep(lines, word, ignore_case):
    """
    Return lines containing a word, ignoring case if asked, with their line
    numbers from 1.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(grep, [
    ((['ok', 'ERROR x', 'error y'], 'error', True), [[2, 'ERROR x'], [3, 'error y']]),
    ((['ok', 'ERROR x'], 'error', False), []),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop with a counter.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A comprehension with `enumerate`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A loop with a counter.

```python
def grep(lines, word, ignore_case):
    out, n = [], 0
    for line in lines:
        n += 1
        a, b = (line.lower(), word.lower()) if ignore_case else (line, word)
        if b in a:
            out.append([n, line])
    return out
```

**Cost:** O(n) time, O(m) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A comprehension with `enumerate`.

```python
def grep(lines, word, ignore_case):
    pattern = re.compile(re.escape(word), re.I if ignore_case else 0)
    return [[i, l] for i, l in enumerate(lines, 1) if pattern.search(l)]
```

**Cost:** O(n) time, O(m) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop with a counter. That costs O(n) time and O(m) space."

"I can do better: A comprehension with `enumerate`. That's O(n) time and O(m) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/engineering-toolbox/build-grep-in-python)

<a id="q2"></a>
### Q2. sort | uniq -c | sort -rn

🟢 Warm-up · *The most useful one-liner*

Return the top k values as [count, value], most frequent first, ties alphabetical.

**Examples**

- `lines = ['b', 'a', 'b', 'c', 'a', 'b'], k = 2` → `[[3, 'b'], [2, 'a']]`

In [ ]:
def top_values(lines, k):
    """
    Return the top k values as [count, value], most frequent first, ties
    alphabetical.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(top_values, [
    ((['b', 'a', 'b', 'c', 'a', 'b'], 2), [[3, 'b'], [2, 'a']]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Sort, then count neighbouring runs, as uniq does.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`Counter`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Sort, then count neighbouring runs, as uniq does.

```python
def top_values(lines, k):
    s = sorted(lines)
    runs = []
    for v in s:
        if runs and runs[-1][1] == v:
            runs[-1][0] += 1
        else:
            runs.append([1, v])
    return sorted(runs, key=lambda r: (-r[0], r[1]))[:k]
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `Counter`.

```python
def top_values(lines, k):
    from collections import Counter
    return sorted(([c, v] for v, c in Counter(lines).items()), key=lambda r: (-r[0], r[1]))[:k]
```

**Cost:** O(n log n) time, O(d) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Sort, then count neighbouring runs, as uniq does. That costs O(n log n) time and O(n) space."

"I can do better: `Counter`. That's O(n log n) time and O(d) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/engineering-toolbox/sort-uniq-c-sort-rn)

<a id="q3"></a>
### Q3. awk: Pick a Column

🟢 Warm-up · *Field extraction*

Sum column index col (0-based) of comma-separated lines, skipping non-numbers.

**Examples**

- `lines = ['a,1', 'b,2', 'c,x'], col = 1` → `3.0`

In [ ]:
def sum_column(lines, col):
    """
    Sum column index col (0-based) of comma-separated lines, skipping non-
    numbers.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(sum_column, [
    ((['a,1', 'b,2', 'c,x'], 1), 3.0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Split and try to convert.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A generator with a regular expression test.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Split and try to convert.

```python
def sum_column(lines, col):
    total = 0.0
    for line in lines:
        try:
            total += float(line.split(",")[col])
        except (ValueError, IndexError):
            pass
    return total
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A generator with a regular expression test.

```python
def sum_column(lines, col):
    number = re.compile(r"-?\d+(\.\d+)?$")
    values = (l.split(",")[col] for l in lines)
    return float(sum(float(v) for v in values if number.match(v)))
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Split and try to convert. That costs O(n) time and O(1) space."

"I can do better: A generator with a regular expression test. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/engineering-toolbox/awk-pick-a-column)

<a id="q4"></a>
### Q4. tail -n

🟢 Warm-up · *Last lines without loading everything*

Return the last n lines of a stream while keeping only n in memory.

**Examples**

- `lines = ['a', 'b', 'c', 'd'], n = 2` → `['c', 'd']`

In [ ]:
def tail(lines, n):
    """
    Return the last n lines of a stream while keeping only n in memory.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(tail, [
    ((['a', 'b', 'c', 'd'], 2), ['c', 'd']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Keep a list, trimming it.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`deque(maxlen=n)`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Keep a list, trimming it.

```python
def tail(lines, n):
    keep = []
    for l in lines:
        keep.append(l)
        if len(keep) > n:
            keep.pop(0)
    return keep
```

**Cost:** O(l·n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `deque(maxlen=n)`.

```python
def tail(lines, n):
    from collections import deque
    return list(deque(lines, maxlen=n))
```

**Cost:** O(l) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Keep a list, trimming it. That costs O(l·n) time and O(n) space."

"I can do better: `deque(maxlen=n)`. That's O(l) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/engineering-toolbox/tail-n)

<a id="q5"></a>
### Q5. Semantic Version Bump

🟢 Warm-up · *Releases*

Bump 'major.minor.patch' for a 'breaking', 'feature' or 'fix' change.

**Examples**

- `version = '1.4.2', change = 'fix'` → `'1.4.3'`
- `version = '1.4.2', change = 'feature'` → `'1.5.0'`
- `version = '1.4.2', change = 'breaking'` → `'2.0.0'`

In [ ]:
def bump(version, change):
    """
    Bump 'major.minor.patch' for a 'breaking', 'feature' or 'fix' change.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(bump, [
    (('1.4.2', 'fix'), '1.4.3'),
    (('1.4.2', 'feature'), '1.5.0'),
    (('1.4.2', 'breaking'), '2.0.0'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Ifs.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A compact conditional.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Ifs.

```python
def bump(version, change):
    major, minor, patch = map(int, version.split("."))
    if change == "breaking":
        major += 1
        minor = 0
        patch = 0
    elif change == "feature":
        minor += 1
        patch = 0
    else:
        patch += 1
    return f"{major}.{minor}.{patch}"
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A compact conditional.

```python
def bump(version, change):
    a, b, c = map(int, version.split("."))
    return {"breaking": f"{a + 1}.0.0", "feature": f"{a}.{b + 1}.0"}.get(change, f"{a}.{b}.{c + 1}")
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Ifs. That costs O(1) time and O(1) space."

"I can do better: A compact conditional. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/engineering-toolbox/semantic-version-bump)

<a id="q6"></a>
### Q6. Environment Variables With Defaults

🟢 Warm-up · *Twelve-factor config*

Given an environment dictionary and required and optional settings, return (config, missing required, sorted).

**Examples**

- `env = {'DB': 'x'}, required = ['DB', 'KEY'], optional = {'PORT': '5432'}` → `({'DB': 'x', 'PORT': '5432'}, ['KEY'])`

In [ ]:
def read_env(env, required, optional):
    """
    Given an environment dictionary and required and optional settings, return
    (config, missing required, sorted).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(read_env, [
    (({'DB': 'x'}, ['DB', 'KEY'], {'PORT': '5432'}), ({'DB': 'x', 'PORT': '5432'}, ['KEY'])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loops.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Comprehensions.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Loops.

```python
def read_env(env, required, optional):
    config, missing = {}, []
    for k in required:
        if k in env:
            config[k] = env[k]
        else:
            missing.append(k)
    for k, default in optional.items():
        config[k] = env.get(k, default)
    return (config, sorted(missing))
```

**Cost:** O(k) time, O(k) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Comprehensions.

```python
def read_env(env, required, optional):
    config = {k: env[k] for k in required if k in env} | {k: env.get(k, d) for k, d in optional.items()}
    return (config, sorted(set(required) - env.keys()))
```

**Cost:** O(k) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loops. That costs O(k) time and O(k) space."

"I can do better: Comprehensions. That's O(k) time and O(k) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/engineering-toolbox/environment-variables-with-defaults)

<a id="q7"></a>
### Q7. Disk Usage Report

🟢 Warm-up · *du in Python*

paths map a file path to bytes. Return total bytes per top-level folder, sorted by folder.

**Examples**

- `paths = {'a/x.log': 10, 'a/b/y.log': 5, 'c/z.log': 7}` → `{'a': 15, 'c': 7}`

In [ ]:
def disk_usage(paths):
    """
    paths map a file path to bytes. Return total bytes per top-level folder,
    sorted by folder.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(disk_usage, [
    (({'a/x.log': 10, 'a/b/y.log': 5, 'c/z.log': 7},), {'a': 15, 'c': 7}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Split on / and add up.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A `defaultdict`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Split on / and add up.

```python
def disk_usage(paths):
    totals = {}
    for p, size in paths.items():
        top = p.split("/")[0]
        totals[top] = totals.get(top, 0) + size
    return dict(sorted(totals.items()))
```

**Cost:** O(n) time, O(d) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A `defaultdict`.

```python
def disk_usage(paths):
    from collections import defaultdict
    totals = defaultdict(int)
    for p, s in paths.items():
        totals[p.partition("/")[0]] += s
    return dict(sorted(totals.items()))
```

**Cost:** O(n) time, O(d) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Split on / and add up. That costs O(n) time and O(d) space."

"I can do better: A `defaultdict`. That's O(n) time and O(d) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/engineering-toolbox/disk-usage-report)

<a id="q8"></a>
### Q8. Log Rotation

🟢 Warm-up · *Don't fill the disk*

Keep files of a pattern 'app.YYYY-MM-DD.log' newer than or equal to keep\_from. Return (kept, deleted), sorted.

**Examples**

- `files = ['app.2026-01-01.log', 'app.2026-01-09.log', 'app.2026-01-05.log'], keep_from = '2026-01-05'` → `(['app.2026-01-05.log', 'app.2026-01-09.log'], ['app.2026-01-01.log'])`

In [ ]:
def rotate(files, keep_from):
    """
    Keep files of a pattern 'app.YYYY-MM-DD.log' newer than or equal to
    keep_from. Return (kept, deleted), sorted.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(rotate, [
    ((['app.2026-01-01.log', 'app.2026-01-09.log', 'app.2026-01-05.log'], '2026-01-05'), (['app.2026-01-05.log', 'app.2026-01-09.log'], ['app.2026-01-01.log'])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Slice the date from the name.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A regular expression, then split.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Slice the date from the name.

```python
def rotate(files, keep_from):
    kept, deleted = [], []
    for f in files:
        if f[4:14] >= keep_from:
            kept.append(f)
        else:
            deleted.append(f)
    return (sorted(kept), sorted(deleted))
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A regular expression, then split.

```python
def rotate(files, keep_from):
    day = lambda f: re.search(r"\d{4}-\d\d-\d\d", f).group()
    return (sorted(f for f in files if day(f) >= keep_from), sorted(f for f in files if day(f) < keep_from))
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Slice the date from the name. That costs O(n log n) time and O(n) space."

"I can do better: A regular expression, then split. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/engineering-toolbox/log-rotation)

<a id="q9"></a>
### Q9. Dependency Pin Check

🟢 Warm-up · *Reproducible builds*

Return requirement lines not pinned with '=='. Comments and blanks are ignored.

**Examples**

- `lines = ['pandas==2.2.0', '# note', '', 'requests>=2', 'numpy']` → `['requests>=2', 'numpy']`

In [ ]:
def unpinned(lines):
    """
    Return requirement lines not pinned with '=='. Comments and blanks are
    ignored.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(unpinned, [
    ((['pandas==2.2.0', '# note', '', 'requests>=2', 'numpy'],), ['requests>=2', 'numpy']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop with checks.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A comprehension.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Loop with checks.

```python
def unpinned(lines):
    out = []
    for l in lines:
        s = l.strip()
        if s and not s.startswith("#") and "==" not in s:
            out.append(s)
    return out
```

**Cost:** O(n) time, O(m) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A comprehension.

```python
def unpinned(lines):
    return [s for s in map(str.strip, lines) if s and s[0] != "#" and "==" not in s]
```

**Cost:** O(n) time, O(m) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop with checks. That costs O(n) time and O(m) space."

"I can do better: A comprehension. That's O(n) time and O(m) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/engineering-toolbox/dependency-pin-check)

<a id="q10"></a>
### Q10. Parse Exit Codes

🟢 Warm-up · *Did the job really succeed?*

Given a list of (command, exit code), return first failing command or None. In a shell script, set -e stops at the first failure.

**Examples**

- `results = [('extract', 0), ('load', 1), ('notify', 0)]` → `'load'`
- `results = [('a', 0)]` → `None`

In [ ]:
def first_failure(results):
    """
    Given a list of (command, exit code), return first failing command or None.
    In a shell script, set -e stops at the first failure.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(first_failure, [
    (([('extract', 0), ('load', 1), ('notify', 0)],), 'load'),
    (([('a', 0)],), None),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`next` with a default.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A loop.

```python
def first_failure(results):
    for cmd, code in results:
        if code != 0:
            return cmd
    return None
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `next` with a default.

```python
def first_failure(results):
    return next((c for c, code in results if code), None)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop. That costs O(n) time and O(1) space."

"I can do better: `next` with a default. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/engineering-toolbox/parse-exit-codes)

<a id="q11"></a>
### Q11. File Permissions

🟢 Warm-up · *chmod 640*

Convert an octal permission string such as '640' to 'rw-r-----'.

**Examples**

- `octal = '640'` → `'rw-r-----'`
- `octal = '755'` → `'rwxr-xr-x'`

In [ ]:
def perms(octal):
    """
    Convert an octal permission string such as '640' to 'rw-r-----'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(perms, [
    (('640',), 'rw-r-----'),
    (('755',), 'rwxr-xr-x'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Check each bit with ifs.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A comprehension over the bits.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Check each bit with ifs.

```python
def perms(octal):
    out = ""
    for digit in octal:
        d = int(digit)
        out += "r" if d & 4 else "-"
        out += "w" if d & 2 else "-"
        out += "x" if d & 1 else "-"
    return out
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A comprehension over the bits.

```python
def perms(octal):
    return "".join(c if int(d) & b else "-" for d in octal for c, b in zip("rwx", (4, 2, 1)))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Check each bit with ifs. That costs O(1) time and O(1) space."

"I can do better: A comprehension over the bits. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/engineering-toolbox/file-permissions)

<a id="q12"></a>
### Q12. Does Cron Match This Time?

🟡 Interview standard · *Scheduling basics*

Given fields 'minute hour day month weekday' with numbers, \* or \*/n, return whether a time matches (weekday 0 = Sunday).

**Examples**

- `expr = '0 2 * * *', minute = 0, hour = 2, day = 5, month = 1, weekday = 3` → `True`
- `expr = '*/15 * * * *', minute = 30, hour = 9, day = 5, month = 1, weekday = 3` → `True`
- `expr = '0 2 * * 1', minute = 0, hour = 2, day = 5, month = 1, weekday = 3` → `False`

In [ ]:
def cron_match(expr, minute, hour, day, month, weekday):
    """
    Given fields 'minute hour day month weekday' with numbers, * or */n, return
    whether a time matches (weekday 0 = Sunday).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(cron_match, [
    (('0 2 * * *', 0, 2, 5, 1, 3), True),
    (('*/15 * * * *', 30, 9, 5, 1, 3), True),
    (('0 2 * * 1', 0, 2, 5, 1, 3), False),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop with ifs.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A small field-matching helper.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A loop with ifs.

```python
def cron_match(expr, minute, hour, day, month, weekday):
    values = [minute, hour, day, month, weekday]
    for field, v in zip(expr.split(), values):
        if field == "*":
            continue
        if field.startswith("*/"):
            if v % int(field[2:]) != 0:
                return False
        elif int(field) != v:
            return False
    return True
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A small field-matching helper.

```python
def cron_match(expr, minute, hour, day, month, weekday):
    def ok(f, v):
        return f == "*" or (v % int(f[2:]) == 0 if f.startswith("*/") else int(f) == v)
    return all(ok(f, v) for f, v in zip(expr.split(), (minute, hour, day, month, weekday)))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop with ifs. That costs O(1) time and O(1) space."

"I can do better: A small field-matching helper. That's O(1) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `expr = '0 2 * * *', minute = 0, hour = 2, day = 5, month = 1, weekday = 3`

1. ok() returns True
2. ok() returns True
3. ok() returns True
4. ok() returns True
5. ok() returns True
6. cron\_match() returns True

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/engineering-toolbox/does-cron-match-this-time)

<a id="q13"></a>
### Q13. Next Cron Runs

🟡 Interview standard · *When will it fire?*

For 'minute hour \* \* \*', return the next n run times after a timestamp, as 'YYYY-MM-DD HH:MM'.

**Examples**

- `minute = 30, hour = 2, after = '2026-01-05 03:00', n = 2` → `['2026-01-06 02:30', '2026-01-07 02:30']`
- `minute = 30, hour = 2, after = '2026-01-05 01:00', n = 1` → `['2026-01-05 02:30']`

In [ ]:
from datetime import datetime, timedelta

def next_runs(minute, hour, after, n):
    """
    For 'minute hour * * *', return the next n run times after a timestamp, as
    'YYYY-MM-DD HH:MM'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(next_runs, [
    ((30, 2, '2026-01-05 03:00', 2), ['2026-01-06 02:30', '2026-01-07 02:30']),
    ((30, 2, '2026-01-05 01:00', 1), ['2026-01-05 02:30']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Step minute by minute. It's simple, but slow.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Jump: today's run if it's still ahead, otherwise tomorrow's.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Step minute by minute. It's simple, but slow.

```python
def next_runs(minute, hour, after, n):
    t = datetime.strptime(after, "%Y-%m-%d %H:%M") + timedelta(minutes=1)
    out = []
    while len(out) < n:
        if t.minute == minute and t.hour == hour:
            out.append(t.strftime("%Y-%m-%d %H:%M"))
        t += timedelta(minutes=1)
    return out
```

**Cost:** O(1440·n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Jump: today's run if it's still ahead, otherwise tomorrow's.

```python
def next_runs(minute, hour, after, n):
    now = datetime.strptime(after, "%Y-%m-%d %H:%M")
    first = now.replace(hour=hour, minute=minute)
    if first <= now:
        first += timedelta(days=1)
    return [(first + timedelta(days=i)).strftime("%Y-%m-%d %H:%M") for i in range(n)]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Step minute by minute. It's simple, but slow. That costs O(1440·n) time and O(n) space."

"I can do better: Jump: today's run if it's still ahead, otherwise tomorrow's. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/engineering-toolbox/next-cron-runs)

<a id="q14"></a>
### Q14. Git: Which Commits Are New?

🟡 Interview standard · *Commits form a graph*

parents maps a commit to its parents. Return commits reachable from branch but not from main, sorted.

**Examples**

- `parents = {'a': [], 'b': ['a'], 'c': ['b'], 'd': ['b'], 'e': ['d']}, main = 'c', branch = 'e'` → `['d', 'e']`

In [ ]:
def new_commits(parents, main, branch):
    """
    parents maps a commit to its parents. Return commits reachable from branch
    but not from main, sorted.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(new_commits, [
    (({'a': [], 'b': ['a'], 'c': ['b'], 'd': ['b'], 'e': ['d']}, 'c', 'e'), ['d', 'e']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Build both ancestor lists, then compare.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Two graph walks with sets.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Build both ancestor lists, then compare.

```python
def new_commits(parents, main, branch):
    def ancestors(x):
        out, todo = [], [x]
        while todo:
            c = todo.pop()
            if c not in out:
                out.append(c)
                todo += parents[c]
        return out
    base = ancestors(main)
    return sorted(c for c in ancestors(branch) if c not in base)
```

**Cost:** O(c²) time, O(c) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Two graph walks with sets.

```python
def new_commits(parents, main, branch):
    def reach(x, seen=None):
        seen = set() if seen is None else seen
        if x not in seen:
            seen.add(x)
            for p in parents[x]:
                reach(p, seen)
        return seen
    return sorted(reach(branch) - reach(main))
```

**Cost:** O(c) time, O(c) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Build both ancestor lists, then compare. That costs O(c²) time and O(c) space."

"I can do better: Two graph walks with sets. That's O(c) time and O(c) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `parents = {'a': [], 'b': ['a'], 'c': ['b'], 'd': ['b'], 'e': ['d']}, main = 'c', branch = 'e'`

1. reach() returns {'b', 'a', 'e', 'd'}
2. reach() returns {'b', 'a', 'e', 'd'}
3. reach() returns {'b', 'a', 'e', 'd'}
4. reach() returns {'b', 'a', 'e', 'd'}
5. reach() returns {'b', 'a', 'c'}
6. reach() returns {'b', 'a', 'c'}
7. reach() returns {'b', 'a', 'c'}
8. new\_commits() returns ['d', 'e']

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/engineering-toolbox/git-which-commits-are-new)

<a id="q15"></a>
### Q15. Docker Layer Cache

🟡 Interview standard · *Why order matters*

Each step is rebuilt if it or any earlier step changed. Given steps with seconds, and the index of the first changed step, return the rebuild seconds. Compare copying code before or after installing packages.

**Examples**

- `steps = [5, 100, 2], first_changed = 2` → `2`
- `steps = [5, 2, 100], first_changed = 1` → `102`

In [ ]:
def rebuild_time(steps, first_changed):
    """
    Each step is rebuilt if it or any earlier step changed. Given steps with
    seconds, and the index of the first changed step, return the rebuild
    seconds. Compare copying code before or after installing packages.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(rebuild_time, [
    (([5, 100, 2], 2), 2),
    (([5, 2, 100], 1), 102),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Add up in a loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A slice.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Add up in a loop.

```python
def rebuild_time(steps, first_changed):
    total = 0
    for i, s in enumerate(steps):
        if i >= first_changed:
            total += s
    return total
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A slice.

```python
def rebuild_time(steps, first_changed):
    return sum(steps[first_changed:])
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Add up in a loop. That costs O(n) time and O(1) space."

"I can do better: A slice. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/engineering-toolbox/docker-layer-cache)

<a id="q16"></a>
### Q16. Find Secrets Before Committing

🟡 Interview standard · *Never commit a key*

Return line numbers (from 1) that look like secrets: a name containing key, secret, token or password assigned a non-empty value.

**Examples**

- `lines = ['name = "x"', 'API_KEY = "abc"', 'password=hunter2', 'key_count = 3', "token = ''"]` → `[2, 3]`

In [ ]:
def find_secrets(lines):
    """
    Return line numbers (from 1) that look like secrets: a name containing key,
    secret, token or password assigned a non-empty value.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(find_secrets, [
    ((['name = "x"', 'API_KEY = "abc"', 'password=hunter2', 'key_count = 3', "token = ''"],), [2, 3]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Split on = and check words.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A regular expression. Real teams use a tool like gitleaks.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Split on = and check words.

```python
def find_secrets(lines):
    out = []
    for i, line in enumerate(lines, 1):
        if "=" not in line:
            continue
        name, value = line.split("=", 1)
        name = name.strip().lower()
        value = value.strip().strip("'\"")
        if value and name.split("_")[-1] in ("key", "secret", "token", "password"):
            out.append(i)
    return out
```

**Cost:** O(n) time, O(m) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A regular expression. Real teams use a tool like gitleaks.

```python
def find_secrets(lines):
    p = re.compile(r"^\s*\w*(key|secret|token|password)\s*=\s*['\"]?[^'\"\s]+", re.I)
    return [i for i, l in enumerate(lines, 1) if p.match(l)]
```

**Cost:** O(n) time, O(m) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Split on = and check words. That costs O(n) time and O(m) space."

"I can do better: A regular expression. Real teams use a tool like gitleaks. That's O(n) time and O(m) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/engineering-toolbox/find-secrets-before-committing)

<a id="q17"></a>
### Q17. Unit Test Runner

🟡 Interview standard · *What pytest does*

Given a function and cases (args, expected), return (passed count, failing case indexes). An exception counts as a failure.

**Examples**

- `cases = [((1, 2), 3), ((1, 1), 3), ((1, 'x'), 0)]` → `(1, [1, 2])`

In [ ]:
def run_tests(cases):
    """
    Given a function and cases (args, expected), return (passed count, failing
    case indexes). An exception counts as a failure.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(run_tests, [
    (([((1, 2), 3), ((1, 1), 3), ((1, 'x'), 0)],), (1, [1, 2])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop with try/except, testing addition.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

The same, with enumerate and a helper.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A loop with try/except, testing addition.

```python
def run_tests(cases):
    def add(a, b):
        return a + b
    passed, failed = 0, []
    for i, (args, want) in enumerate(cases):
        try:
            if add(*args) == want:
                passed += 1
            else:
                failed.append(i)
        except TypeError:
            failed.append(i)
    return (passed, failed)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** The same, with enumerate and a helper.

```python
def run_tests(cases):
    def add(a, b):
        return a + b

    def ok(args, want):
        try:
            return add(*args) == want
        except TypeError:
            return False
    results = [ok(*c) for c in cases]
    return (sum(results), [i for i, r in enumerate(results) if not r])
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop with try/except, testing addition. That costs O(n) time and O(n) space."

"I can do better: The same, with enumerate and a helper. That's O(n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `cases = [((1, 2), 3), ((1, 1), 3), ((1, 'x'), 0)]`

1. add() returns 3
2. ok() returns True
3. add() returns 2
4. ok() returns False
5. TypeError raised: unsupported operand type(s) for +: 'int' and 'str'
6. add() returns None
7. TypeError raised: unsupported operand type(s) for +: 'int' and 'str'
8. ok() returns False
9. run\_tests() returns (1, [1, 2])

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/engineering-toolbox/unit-test-runner)

<a id="q18"></a>
### Q18. Mock an External Call

🟡 Interview standard · *Test without the network*

A load function takes a fetch function. Pass a fake fetch returning rows, and return the rows it loads with amount above 0. Injecting dependencies makes code testable.

**Examples**

- `fake_rows = [{'a': 1}, {'a': -1}]` → `[{'a': 1}]`

In [ ]:
def load_with_fake(fake_rows):
    """
    A load function takes a fetch function. Pass a fake fetch returning rows,
    and return the rows it loads with amount above 0. Injecting dependencies
    makes code testable.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(load_with_fake, [
    (([{'a': 1}, {'a': -1}],), [{'a': 1}]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A fake function in the test.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A lambda as the stub.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A fake function in the test.

```python
def load_with_fake(fake_rows):
    def fake_fetch():
        return fake_rows

    def load(fetch):
        return [r for r in fetch() if r["a"] > 0]
    return load(fake_fetch)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A lambda as the stub.

```python
def load_with_fake(fake_rows):
    load = lambda fetch: [r for r in fetch() if r["a"] > 0]
    return load(lambda: fake_rows)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A fake function in the test. That costs O(n) time and O(n) space."

"I can do better: A lambda as the stub. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/engineering-toolbox/mock-an-external-call)

<a id="q19"></a>
### Q19. CI Pipeline Order

🟡 Interview standard · *Fail fast*

Given stages with (name, seconds, failure chance), order by seconds ascending and return the order and the expected seconds until stopping, rounded to 2 decimals. Each stage runs only if earlier ones passed.

**Examples**

- `stages = [('tests', 60, 0.2), ('lint', 5, 0.1)]` → `(['lint', 'tests'], 59.0)`

In [ ]:
def ci_order(stages):
    """
    Given stages with (name, seconds, failure chance), order by seconds
    ascending and return the order and the expected seconds until stopping,
    rounded to 2 decimals. Each stage runs only if earlier ones passed.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(ci_order, [
    (([('tests', 60, 0.2), ('lint', 5, 0.1)],), (['lint', 'tests'], 59.0)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop tracking the chance of reaching each stage.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same, compactly.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A loop tracking the chance of reaching each stage.

```python
def ci_order(stages):
    ordered = sorted(stages, key=lambda s: s[1])
    reach, expected = 1.0, 0.0
    for name, seconds, fail in ordered:
        expected += reach * seconds
        reach *= 1 - fail
    return ([s[0] for s in ordered], round(expected, 2))
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Same, compactly.

```python
def ci_order(stages):
    ordered = sorted(stages, key=lambda s: s[1])
    reach, cost = 1.0, 0.0
    for _, sec, fail in ordered:
        cost, reach = cost + reach * sec, reach * (1 - fail)
    return ([s[0] for s in ordered], round(cost, 2))
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop tracking the chance of reaching each stage. That costs O(n log n) time and O(n) space."

"I can do better: Same, compactly. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/engineering-toolbox/ci-pipeline-order)

<a id="q20"></a>
### Q20. Merge Conflict Check

🟡 Interview standard · *Three-way merge*

Given base, ours and theirs for lines at the same positions: take whichever changed; if both changed differently, it's a conflict. Return (merged with 'CONFLICT' where needed, number of conflicts).

**Examples**

- `base = ['a', 'b', 'c'], ours = ['a', 'B', 'c'], theirs = ['a', 'b', 'C']` → `(['a', 'B', 'C'], 0)`
- `base = ['a'], ours = ['x'], theirs = ['y']` → `(['CONFLICT'], 1)`

In [ ]:
def merge3(base, ours, theirs):
    """
    Given base, ours and theirs for lines at the same positions: take whichever
    changed; if both changed differently, it's a conflict. Return (merged with
    'CONFLICT' where needed, number of conflicts).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(merge3, [
    ((['a', 'b', 'c'], ['a', 'B', 'c'], ['a', 'b', 'C']), (['a', 'B', 'C'], 0)),
    ((['a'], ['x'], ['y']), (['CONFLICT'], 1)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Ifs per line.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A helper in a comprehension.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Ifs per line.

```python
def merge3(base, ours, theirs):
    out, conflicts = [], 0
    for b, o, t in zip(base, ours, theirs):
        if o == t or t == b:
            out.append(o)
        elif o == b:
            out.append(t)
        else:
            out.append("CONFLICT")
            conflicts += 1
    return (out, conflicts)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A helper in a comprehension.

```python
def merge3(base, ours, theirs):
    pick = lambda b, o, t: o if o == t or t == b else t if o == b else "CONFLICT"
    out = [pick(*x) for x in zip(base, ours, theirs)]
    return (out, out.count("CONFLICT"))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Ifs per line. That costs O(n) time and O(n) space."

"I can do better: A helper in a comprehension. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/course/#/q/engineering-toolbox/merge-conflict-check)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. What does set -euo pipefail do?</b></summary>

Stops a script on an error (-e), on an unset variable (-u), and when any part of a pipeline fails (pipefail). Without it, failures can pass silently.

</details>

<details>
<summary><b>2. Merge vs rebase?</b></summary>

Merge keeps history as it happened, adding a merge commit. Rebase replays your commits on top of another branch, giving a straight history. Don't rebase commits others are already using.

</details>

<details>
<summary><b>3. Container vs virtual machine?</b></summary>

A VM includes a full operating system. A container shares the host kernel and packages only the app and its dependencies, so it's lighter and starts faster.

</details>

<details>
<summary><b>4. Why do you put dependency install before copying code in a Dockerfile?</b></summary>

Layers are cached, and a change rebuilds everything after it. Code changes often and dependencies rarely, so this keeps most builds fast.

</details>

<details>
<summary><b>5. How do you handle secrets?</b></summary>

Use a secrets manager or injected environment variables, never Git or images. If a key leaks, rotate it, since Git history retains it.

</details>

<details>
<summary><b>6. How do you test a data pipeline?</b></summary>

Unit tests for transforms with small fixtures and edge cases, integration tests on a test database, and data quality checks on real runs.

</details>

<details>
<summary><b>7. Scenario: disk full on a server. What do you do?</b></summary>

Use df -h to confirm, then du -sh \* to find what's large. Often it's old logs or temp files. Clear them safely, then add rotation and an alert so that it doesn't recur.

</details>

<details>
<summary><b>8. Scenario: a script reports success, but the output file is empty. What might be wrong?</b></summary>

An earlier step probably failed and the script carried on, or a pipeline hid the error. Add set -euo pipefail, and check row counts as a final step.

</details>

<details>
<summary><b>9. Scenario: you committed an API key by accident. What now?</b></summary>

Rotate the key immediately, since it must be treated as compromised. Then remove it from the code, and from history if needed, and add a secret scanner to CI.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: The terminal: your daily tool
- [ ] I can explain: Scheduling with cron
- [ ] I can explain: Git: how teams work on code
- [ ] I can explain: Docker: the same everywhere
- [ ] I can explain: Testing and CI/CD

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Find Secrets Before Committing
- [ ] [Q17](#q17) Unit Test Runner
- [ ] [Q18](#q18) Mock an External Call
- [ ] [Q19](#q19) CI Pipeline Order
- [ ] [Q20](#q20) Merge Conflict Check

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.